# 4. SeqTO-v1: The C-Core MDP Environment

## 4.1 Physical System

A C-core electromagnetic actuator consists of a ferromagnetic core shaped like the letter C, two excitation coils wound around the core legs, and a movable iron armature that bridges the air gap. When current flows through the coils, magnetic flux circulates through the core and armature, generating an attractive force that pulls the armature into the gap.

**Design objective:** maximise the magnetic force on the armature by optimising the iron distribution in the design domain.  
Force scales with flux density:

$$F \approx \frac{B^2 A_{\text{arm}}}{2\mu_0}$$

Maximising the mean $|B|$ at the six armature sample points therefore directly maximises force.

**Baseline:** A conventional C-core achieves **40 N**. Published ON/OFF topology optimisation (Midha 2019) reaches **50.9 N** with post-processing filters. The SeqTO-v1 + GA approach presented in the thesis achieves **51 N** (non-linear M-19 steel, 3×3 controller) and **70 N** (linear pure iron, 3×3 controller).

---

## 4.2 Design Domain

The full simulation grid is **18 rows × 35 columns** (mm), giving a 1 mm × 1 mm resolution. The optimisable **design window** occupies rows 3–17 × cols 5–25, i.e. **15 × 21 = 315 cells** (27 excluded for right-coil overlap → **288 designable cells**).

| Region | Rows | Cols | Role |
|---|---|---|---|
| Left coil | 0–8 | 2–4 | Fixed copper, −500 A-turns |
| Right coil | 0–8 | 11–13 | Fixed copper, +500 A-turns |
| Armature | 0–14 | 27–32 | Fixed iron, force measurement target |
| **Design window** | **3–17** | **5–25** | **288 designable cells** (315 window − 27 right-coil exclusion, rows 0–8 × cols 11–13) |
| Buffer rows | 0–2 | — | Non-design boundary |

The controller deposits iron cells as it traverses the design window; it cannot enter the coil or armature regions. The material budget is constrained to $K_1 = 80 \leq n_{\text{iron}} \leq K_2 = 180$ cells.


```{figure} ../_static/figures/c_core_full.png
---
name: fig-ccore-full
width: 75%
---
C-core electromagnetic actuator. Two copper coils generate magnetic flux that circulates through the ferromagnetic core and across the air gap to the movable iron armature.
```


In [ ]:
"""
C-core design domain layout — pure Python, no FEMM required.

Design window: rows 0-14, cols 5-25.
  Starts at row 0 (same as coils and armature top).
  Ends at row 14 (same as armature bottom, ARM_ROWS[1]=15).
  Right-coil exclusion (rows 0-8, cols 11-13) = 27 cells excluded → 288 designable cells.
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# -- Constants (mirror seqTO_env.py) ----------------------------------------
ROWS, COLS  = 18, 35
COIL_LEN    = 9
COIL1_COLS  = (2, 5)
COIL2_COLS  = (11, 14)
ARM_ROWS    = (0, 15)
ARM_COLS    = (27, 33)
DW_ROWS     = (0, ARM_ROWS[1])   # (0, 15)
DW_COLS     = (5, 26)
K1, K2      = 80, 180

def is_design_cell(i, j):
    in_dw    = DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]
    in_coil2 = i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]
    return in_dw and not in_coil2

# -- Build region grid (integer code per cell) --------------------------------
# 0=other  1=left-coil  2=right-coil  3=armature
# 4=designable  5=non-design  6=right-coil excl. inside DW
grid = np.zeros((ROWS, COLS), dtype=int)
for i in range(ROWS):
    for j in range(COLS):
        if i < COIL_LEN and COIL1_COLS[0] <= j < COIL1_COLS[1]:
            grid[i, j] = 1
        elif i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]:
            grid[i, j] = 2
        elif ARM_ROWS[0] <= i < ARM_ROWS[1] and ARM_COLS[0] <= j < ARM_COLS[1]:
            grid[i, j] = 3
        elif is_design_cell(i, j):
            grid[i, j] = 4
        elif DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]:
            grid[i, j] = 6   # right-coil exclusion inside DW
        else:
            grid[i, j] = 5

n_design = int(np.sum(grid == 4))

# -- Plot -----------------------------------------------------------------------
CLRS = ['#F5F5F5', '#2980B9', '#E67E22', '#27AE60',
        '#FEF9E7', '#D5D8DC', '#FAD7A0']

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(grid, cmap=ListedColormap(CLRS), vmin=0, vmax=6,
          origin='upper', aspect='equal')

for i in range(ROWS + 1):
    ax.axhline(i - 0.5, color='#999', lw=0.2, alpha=0.5)
for j in range(COLS + 1):
    ax.axvline(j - 0.5, color='#999', lw=0.2, alpha=0.5)

# Design window boundary
ax.add_patch(mpatches.Rectangle(
    (DW_COLS[0]-0.5, DW_ROWS[0]-0.5), DW_COLS[1]-DW_COLS[0], DW_ROWS[1]-DW_ROWS[0],
    fill=False, ec='#2C3E50', lw=2, ls='--'))

# Controller initial 3×3 block
init_r, init_c = DW_ROWS[0]+1, DW_COLS[0]+1
ax.add_patch(mpatches.Rectangle(
    (init_c-1-0.5, init_r-1-0.5), 3, 3,
    fill=False, ec='crimson', lw=2))
ax.text(init_c, init_r, '3\u00d73\nctrl', ha='center', va='center',
        fontsize=7, color='crimson', fontweight='bold')

# Region labels
ax.text(3,  4, '\u2212500\nturns', ha='center', va='center',
        fontsize=8, color='white', fontweight='bold')
ax.text(12, 4, '+500\nturns', ha='center', va='center',
        fontsize=8, color='white', fontweight='bold')
ax.text(30, 6.5, 'Armature\n(force\ntarget)', ha='center', va='center',
        fontsize=8, color='white', fontweight='bold')
ax.text(12, 12, f'Design window  ({n_design} designable cells)',
        ha='center', va='center', fontsize=9, color='#2C3E50',
        style='italic', fontweight='bold')
ax.text(12, 2, 'excl.', ha='center', va='center',
        fontsize=7, color='#7D6608')

ax.set_xlabel('column (mm)'); ax.set_ylabel('row (mm)')
ax.set_xticks(range(0, COLS, 5)); ax.set_yticks(range(0, ROWS, 3))
ax.set_title(
    f'C-core SeqTO-v1 design domain  |  {ROWS}\u00d7{COLS} mm grid  |  '
    f'{n_design} designable cells  |  3\u00d73 controller  |  '
    f'budget K\u2081={K1}\u2013K\u2082={K2} cells',
    fontsize=11)
ax.legend(handles=[
    mpatches.Patch(fc='#2980B9', label='Left coil  (rows 0\u20138, cols 2\u20134, \u2212500 A-turns)'),
    mpatches.Patch(fc='#E67E22', label='Right coil (rows 0\u20138, cols 11\u201313, +500 A-turns)'),
    mpatches.Patch(fc='#27AE60', label='Armature   (rows 0\u201314, cols 27\u201332, group=5)'),
    mpatches.Patch(fc='#FEF9E7', ec='#2C3E50', lw=0.8,
                   label=f'Designable cells ({n_design}, right-coil excl.)'),
    mpatches.Patch(fc='#FAD7A0', label='Right-coil exclusion (27 cells, rows 0\u20138 \u00d7 cols 11\u201313)'),
    mpatches.Patch(fc='#D5D8DC', label='Buffer / non-design (rows 15\u201317)'),
    mpatches.Patch(fc='none', ec='crimson', label='Controller initial position (3\u00d73)'),
], loc='lower right', fontsize=8, framealpha=0.92)
plt.tight_layout(); plt.show()

print(f'  Design window   : rows 0\u201314, cols 5\u201325  (= ARM_ROWS extent)')
print(f'  Designable cells: {n_design}  (= 15\u00d721 window \u2212 27 right-coil overlap)')
print(f'  Controller start: row={init_r}, col={init_c}')
print(f'  Budget          : K\u2081={K1} \u2264 iron cells \u2264 K\u2082={K2}')


---

## 4.3 SeqTO-v1 MDP Formulation

The C-core topology optimisation problem is cast as a Markov Decision Process (MDP). A 3×3 controller traverses the 18×35 mm design domain, depositing iron cells one step at a time. After each deposit, a full FEMM finite-element solve computes the reward.

| MDP Element | Definition |
|---|---|
| **State** $s_t$ | Flattened iron mask — 18×35 binary grid (630 values) |
| **Action** $a_t$ | 4 discrete moves: 0 = RIGHT, 1 = LEFT, 2 = UP, 3 = DOWN |
| **Transition** | Controller moves; 3×3 block centred at new position becomes iron |
| **Reward** $R_t$ | $\text{mean}(|B|) \times 1000$ at 6 armature sample points (FEMM FEA every step) |
| **Terminal** | `step ≥ max_steps` OR `iron_count > K2 = 180` |
| **Material** | Cold rolled low carbon strip steel (linear B-H for tractable FEMM solve) |
| **Episode length** | 34 steps (C-path demonstration length) |
| **Excitation** | 1 A coil current, 500 turns per coil, 40 mm stack depth |

**Why reward at every step?**  
Each FEMM solve takes ≈0.5 s. Providing dense reward gives the agent immediate feedback on each 3×3 iron deposit, which is critical during early episodes when the design domain is nearly empty and sparse rewards would provide no learning signal.

**Out-of-bounds and coil-collision moves** are silently rejected: the controller stays at its previous position and the iron mask is unchanged. A FEMM solve still runs on the unchanged mask, so the episode continues without interruption.


---

## 4.4 Environment Implementation — `seqTO_env.py`

The full environment is implemented in `seqTO_env.py`. Key geometry constants match the thesis C-core specification exactly:


In [ ]:
# ── Geometry constants (seqTO_env.py) ────────────────────────────────────────
ROWS, COLS = 18, 35          # full simulation grid (mm)
BUFFER     = 0               # no top buffer — design window starts at row 0

COIL_LEN   = 9
COIL1_COLS = (2, 5)          # left coil  cols 2-4  (−500 turns)
COIL2_COLS = (11, 14)        # right coil cols 11-13 (+500 turns)

ARM_ROWS   = (0, 15)         # armature rows 0-14
ARM_COLS   = (27, 33)        # armature cols 27-32  (group=5 for force)

DW_ROWS    = (0, 15)         # design window rows: 0-14   (15 rows, aligned with armature)
DW_COLS    = (5, 26)         # design window cols: 5-25   (21 cols)
                              # 15×21=315 window − 27 right-coil exclusion = 288 designable cells

K1 = 80    # minimum iron cells (material budget lower bound)
K2 = 180   # maximum iron cells (conventional C-core ≈ 180 cells)
MAX_STEPS = 25    # default maximum steps per training episode

FEMM_DEPTH    = 40.0          # mm out-of-plane stack depth
BASE_FEM_FILE = 'ccore_seqto.fem'
DESIGNS_DIR   = 'designs'     # output folder for bitmaps and Q-tables
IRON_MATERIAL = 'Cold rolled low carbon strip steel'

# Actions: (name, Δrow, Δcol)
ACTIONS = {
    0: ('RIGHT',  0, +1),
    1: ('LEFT',   0, -1),
    2: ('UP',    -1,  0),
    3: ('DOWN',  +1,  0),
}

The `CcoreSeqTOEnv` class implements the MDP interface:

In [ ]:
# ── CcoreSeqTOEnv: key methods (abbreviated) ──────────────────────────────────

class CcoreSeqTOEnv:
    """
    SeqTO-v1 MDP for C-core topology optimisation.
    State   : flattened iron_mask  (630 float values, 18×35 grid)
    Actions : 0=RIGHT  1=LEFT  2=UP  3=DOWN
    Reward  : mean |B| at 6 armature points × 1000  (FEMM FEA every step)
    Terminal: step ≥ max_steps  OR  iron_count > K2
    """

    def reset(self):
        """Clear iron mask, return to start position, return initial state."""
        self.iron_mask[:] = 0
        self.pos_r = DW_ROWS[0] + 2    # top-left of design window + 2
        self.pos_c = DW_COLS[0] + 2
        self.step_count = 0
        self.done = False
        return self.iron_mask.flatten().astype(float)

    def step(self, action, save_bmp=None):
        """Move controller, deposit 3×3 iron block, get FEMM reward.
        save_bmp: optional path for B-field bitmap (passed to calculate_reward)."""
        name, dr, dc = ACTIONS[action]
        new_r, new_c = self.pos_r + dr, self.pos_c + dc

        issue = self._check_position(new_r, new_c)  # 0=valid, 1=outside DW, 2=coil collision
        if issue == 0:
            self.pos_r, self.pos_c = new_r, new_c
            for di in range(-1, 2):                    # 3×3 deposit
                for dj in range(-1, 2):
                    ri, ci = self.pos_r + di, self.pos_c + dj
                    if is_design_cell(ri, ci):
                        self.iron_mask[ri, ci] = 1

        self.step_count += 1
        iron_count = int(np.sum(self.iron_mask))
        self.done = (self.step_count >= self.max_steps) or (iron_count > K2)

        reward, force, _ = calculate_reward(self.iron_mask, save_bmp=save_bmp)
        info = {'step': self.step_count, 'action': name,
                'iron_count': iron_count, 'force_N': force, 'issue': issue}
        return self.iron_mask.flatten().astype(float), reward, self.done, info

The FEMM reward is computed by `calculate_reward(iron_mask)`, which:
1. Opens the base `.fem` geometry file (built once by `build_base_geometry()`)
2. Resets all design cells to Air, then applies the current `iron_mask`
3. Saves to a throw-away temp file (`_tmp_solve.fem`) so the master geometry is never overwritten by `mi_analyse`
4. Runs `mi_analyse(0)` → `mi_loadsolution()` to solve
5. Probes $B_x, B_y$ at 6 armature points: $(x=0.2,\ y=27.5)$ to $(x=0.2,\ y=32.5)$
6. Returns $\text{reward} = (\sum|B|/6) \times 1000$

```{note}
The temp-file approach (`mi_saveas` before `mi_analyse`) is critical for robustness across thousands of sequential FEMM calls. Without it, `mi_analyse(0)` overwrites the master `.fem` file on every call, accumulating node-position drift that eventually causes Triangle (the FEMM mesher) to fail with "problem loading mesh".
```


---

## 4.5 C-Path Demonstration — `demo_cpath.py`

Before training any RL agent, a hardcoded **C-shaped path** is used to verify that the environment produces physically meaningful results. The path traces the canonical C-core flux circuit: down the left column, across the bottom, up the right side adjacent to the armature.

```python
# C-path action sequence: 34 steps total
# DOWN 8 → pos_r: 3→11  (left vertical bar)
# RIGHT 18 → pos_c: 5→23  (bottom horizontal bar)
# UP 8 → pos_r: 11→3  (right vertical bar, adjacent to armature)
C_PATH_ACTIONS = [3]*8 + [0]*18 + [2]*8
```

After every step, three files are saved:
- `step_NN_trajectory.png` — matplotlib layout showing iron cells labeled by step number and controller 3×3 outline
- `step_NN_geometry.emf` — FEMM preprocessor geometry view (Enhanced Metafile)
- `step_NN_field.bmp` — FEMM B-field density plot from `mo_savebitmap`

The reward rises monotonically as the iron path grows — confirming that the FEMM reward is correctly capturing the increasing flux in the armature region as the flux circuit is progressively completed.


```{figure} ../content/designs/step_34_trajectory.png
---
name: fig-cpath-trajectory
width: 100%
---
C-path demonstration: final state after 34 steps. Iron cells (grey) form a C-shaped flux circuit — left vertical leg, bottom bar, and right vertical leg adjacent to the armature. Each 3×3 block is labeled with its step number. The black border shows the controller position at the end of the episode.
```


---

## 4.6 State Representation in Tabular Q-Learning

Before training any RL agent, a fundamental design choice must be made: **what information constitutes the state?** Two formulations are compared empirically to show that this choice alone determines whether Q-learning converges.

### Representation 1 — Controller Position Only

$$s_t = (\text{row}_t,\ \text{col}_t) \in \{3,\ldots,17\} \times \{5,\ldots,25\}$$

| Property | Value |
|---|---|
| State space | $15 \times 21 = 315$ states |
| Q-table | Dense NumPy array, shape $(15, 21, 4)$, **1,260 entries** |
| Memory | ≈ 10 KB |
| State revisitability | High — same (row, col) visited many times across episodes |

The key property enabling convergence is **state revisitability**: the controller returns to the same (row, col) in multiple episodes, so each Q-value receives many updates. The trade-off is **information loss**: the agent has no knowledge of what iron has already been deposited, so the Markov property only holds approximately.

### Representation 2 — Full Iron Mask

$$s_t = \text{vec}(\mathbf{M}_t) \in \{0, 1\}^{630}$$

| Property | Value |
|---|---|
| State space | $2^{630} \approx 10^{190}$ states |
| Q-table | Sparse Python `defaultdict` keyed by 630-element tuple |
| Memory | Grows ≈100 KB per 100 episodes, unbounded |
| State revisitability | Effectively zero — iron mask grows monotonically |

The fundamental problem is **state non-revisitability**: because iron cells only go from Air to Iron (never back), the specific path taken determines the exact configuration at each step, making it essentially impossible to revisit any previously-seen state. After $N$ episodes of $T$ steps, the Q-dictionary has $\sim N \times T$ entries but virtually every Q-value has been updated exactly once — from its initialised zero. A single-update Q-value carries no more information than a random number.


In [ ]:
# ── Excerpt from q_learn_position.py ────────────────────────────────────────────
ALPHA, GAMMA = 0.1, 0.9  # hyperparams — defined in the full script
import numpy as np

R_RANGE = 15    # DW_ROWS[1] - DW_ROWS[0]
C_RANGE = 21    # DW_COLS[1] - DW_COLS[0]
Q = np.zeros((R_RANGE, C_RANGE, 4))   # 1,260 entries, ~10 KB

def get_state_position(env):
    """Zero-based index within the design domain."""
    return (env.pos_r - DW_ROWS[0], env.pos_c - DW_COLS[0])

def q_update_position(r, c, action, reward, nr, nc, done):
    target = reward if done else reward + GAMMA * np.max(Q[nr, nc, :])
    Q[r, c, action] += ALPHA * (target - Q[r, c, action])

In [ ]:
# ── Excerpt from q_learn_fullstate.py ───────────────────────────────────────────
ALPHA, GAMMA = 0.1, 0.9  # hyperparams — defined in the full script
from collections import defaultdict

Q_full = defaultdict(lambda: np.zeros(4))   # key = 630-element tuple

def get_state_full(env):
    """Hashable 630-element binary tuple of the current iron mask."""
    return tuple(env.iron_mask.flatten().astype(int))

def q_update_full(state, action, reward, next_state, done):
    target = reward if done else reward + GAMMA * np.max(Q_full[next_state])
    Q_full[state][action] += ALPHA * (target - Q_full[state][action])

### Q-dictionary growth — why full-state fails

| Episodes | Approximate Q entries | Mean updates per entry |
|---|---|---|
| 10 | ≈340 | ≈1 |
| 50 | ≈1,700 | ≈1 |
| 1,000 | ≈34,000 | ≈1 |
| ∞ | unbounded | ≈1 |

The dictionary grows linearly with episodes but the update density stays at 1. No convergence is possible within any practical episode budget.


---

## 4.7 Training and Results

> **Note:** The results below were produced with the pre-correction environment (`DW_ROWS=(3,18)`, 297 cells). After correcting the design window to `DW_ROWS=(0,15)` (288 cells), retraining is required. Numbers will differ.

Both Q-learning variants are trained for **200 episodes × 34 steps** (6,800 FEMM evaluations each) with identical hyperparameters:

| Parameter | Value |
|---|---|
| Episodes | 200 |
| Steps / episode | 34 |
| Learning rate α | 0.1 |
| Discount γ | 0.9 |
| ε start → min | 1.0 → 0.05 |
| ε decay | 0.97 per episode |

Run with:
```bash
python q_learn_position.py    # converges; saves Q-table + learning curve + policy map
python q_learn_fullstate.py   # does not converge; saves Q-dict growth plot
```

**Position-only results (200 episodes):**
- Episode reward range: 16,951 – 26,114 (cumulative over 34 steps)
- Per-step mean |B|: 0.499 – 0.768 T  (= episode reward / 34 / 1000)
- Non-zero Q-entries after training: 633 / 1,260 (50.2%); 274/315 states visited
- Greedy action distribution: RIGHT 42.3%, LEFT 20.8%, UP 19.7%, DOWN 17.2%


```{figure} ../content/designs/q_position/learning_curve_position.png
---
name: fig-q-learning-curve
width: 90%
---
Position-only tabular Q-learning: cumulative episode reward over 200 episodes. High variance during early exploration (ε = 1.0 → 0.05 by episode ~97) reflects diverse random trajectories; rewards range from ~17,000 to ~26,000. After ε reaches its minimum (0.05), the late-episode baseline of ~18,617 recurs repeatedly — the agent deterministically repeating the same suboptimal path under near-greedy policy. This 'worse than random' signature is the empirical fingerprint of a representation mismatch: position-only Q-values aggregate over many topological contexts and produce value estimates that actively degrade performance under exploitation.
```


```{figure} ../content/designs/q_position/policy_position.png
---
name: fig-q-policy-map
width: 90%
---
Greedy policy after 50 training episodes (colour = max Q-value). Meaningful Q-values (blue–yellow) are concentrated in the top-left ~5-column region — the portion of the design domain reachable within a 34-step episode starting from position (0,0). The right 75% of the grid (columns 5–20) remains entirely at Q = 0 (dark purple), with the greedy action defaulting to RIGHT (action 0) by argmax convention.
```


---

## 4.8 Convergence Comparison

| Property | Position-only | Full iron\_mask |
|---|---|---|
| State space size | 315 | $\approx 10^{190}$ |
| Q-table structure | Dense array 15×21×4 | Sparse defaultdict |
| Memory | ≈10 KB | Grows ≈100 KB / 100 episodes |
| State revisitability | High — same position visited every episode | Effectively zero |
| Updates per Q-entry (200 ep) | ≈(200 × avg\_visits) | ≈1 |
| Convergence | Flat reward band within 20–50 episodes | None within any practical budget |
| Information content | Position only (Markov approximation) | Full topology (exact Markov) |
| Practical tractability | Yes | No |

### Key Takeaway

> **The state representation determines the effective size of the learning problem.** Choosing position-only reduces a $2^{630}$-state problem to a 315-state problem — compression that makes Q-learning tractable at the cost of discarding topology history. The full-state formulation is correct in principle but requires function approximation to be usable in practice.

This motivates the natural progression of state representations:

1. **Position only** — tabular Q-learning, tractable, ignores topology history *(this chapter)*
2. **Full iron\_mask** — tabular Q-learning, theoretically exact, practically intractable *(this chapter)*
3. **Full iron\_mask + position** — Deep Q-Network (DQN), generalises across similar states, tractable *(Chapter 5)*

The flat learning curve from the position-only agent and the zero-convergence of the full-state agent together provide empirical justification for moving to Deep RL in Chapter 5.


---

## 4.9 Comparison to Thesis Best Results

The tabular Q-learning experiments here are tractability demonstrations, not performance benchmarks. The thesis best results (from GA and TD-learning on the full SeqTO-v1 environment) are:

| Method | Material | Force | FE calls |
|---|---|---|---|
| Conventional C-core | — | 40 N | — |
| ON/OFF + GA (Midha 2019) | M-19 (non-linear) | 50.9 N | 45,000 |
| SeqTO-v1 + GA | M-19 (non-linear, 3×3) | **51 N** | 7,200 |
| SeqTO-v1 + GA | Linear iron (3×3) | **70 N** | 4,500 |
| SeqTO-v1 + TD-learning | M-19 (non-linear, 3×3) | **51 N** | 6,000 |
| SeqTO-v1 + TD-learning | Linear iron (3×3) | **70 N** | 3,750 |
| **Position-only tabular Q (this notebook)** | Cold rolled steel | — | **6,800** |

The tabular Q agent uses only 6,800 FE calls across 200 episodes — far fewer than any GA or TD method — yet produces no improving policy because the position-only state is insufficient. This confirms that the representation choice, not the computation budget, is the primary determinant of learning quality.

The full production environment used for TD-learning and DRL experiments is `CcoreFemmEnv` in `thesis_code/Chp5_DRL_TO/src/femm_environment/environment.py` (Chapter 5). It uses the same physical geometry and FEMM reward mechanics but extends the action space to 8 directions and adds a B-field observation channel.
